|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>The async engine<h1>|
|<h2>Lecture:</h2>|<h1><b>One loop, two clocks, and what happens when a client hangs up<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import asyncio
import time

import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

STEP_S = 0.010        # one engine step, 10 ms
N       = 8           # concurrent clients
TOKENS  = 20          # tokens each

# The loop that must never block

Everything so far was one process doing one thing. A server is two things at
once: HTTP and a GPU, and neither is allowed to wait for the other.

Get it wrong in the obvious way and every request in flight pays for every
other one.

In [ ]:
RUNNERS = """async def run_blocking(n=N, tokens=TOKENS):
  \"\"\"The model runs inside the event loop. Nothing else can.\"\"\"
  lat, T0 = [], time.perf_counter()
  async def client(i):
    first = None
    for _ in range(tokens):
      time.sleep(STEP_S)                       # blocking call
      if first is None: first = time.perf_counter() - T0
    lat.append((first, time.perf_counter() - T0))
  await asyncio.gather(*[client(i) for i in range(n)])
  return lat, time.perf_counter() - T0

async def run_threaded(n=N, tokens=TOKENS):
  \"\"\"The model runs in a thread pool. The event loop stays responsive.\"\"\"
  loop = asyncio.get_running_loop()
  lat, T0 = [], time.perf_counter()
  async def client(i):
    first = None
    for _ in range(tokens):
      await loop.run_in_executor(None, time.sleep, STEP_S)
      if first is None: first = time.perf_counter() - T0
    lat.append((first, time.perf_counter() - T0))
  await asyncio.gather(*[client(i) for i in range(n)])
  return lat, time.perf_counter() - T0

async def run_engine(n=N, tokens=TOKENS):
  \"\"\"ONE loop steps the model. Clients read tokens off their own queue.\"\"\"
  qs = {i: asyncio.Queue() for i in range(n)}
  lat, T0 = [], time.perf_counter()

  async def engine():
    for k in range(tokens):
      await asyncio.sleep(STEP_S)              # one step serves EVERYBODY
      for q in qs.values(): q.put_nowait(k)

  async def client(i):
    first = None
    for _ in range(tokens):
      await qs[i].get()
      if first is None: first = time.perf_counter() - T0
    lat.append((first, time.perf_counter() - T0))

  await asyncio.gather(engine(), *[client(i) for i in range(n)])
  return lat, time.perf_counter() - T0"""

In [ ]:
exec(RUNNERS)

# notebooks already run an event loop, so `await` at top level rather
# than asyncio.run(), which refuses to nest.
print(f"{'design':>10} {'wall':>7} {'TTFT p50':>10} {'TTFT p99':>10} {'last done':>10}")
results = {}
for name, fn in (('blocking', run_blocking), ('threaded', run_threaded), ('engine', run_engine)):
  lat, wall = await fn()
  ttft = sorted(a for a,_ in lat)
  done = max(b for _,b in lat)
  results[name] = (wall, ttft, done)
  print(f'{name:>10} {wall:>6.2f}s {ttft[len(ttft)//2]:>9.3f}s {ttft[-1]:>9.3f}s {done:>9.2f}s')

### Read the first row

Eight clients, twenty tokens each, ten milliseconds a step. The work is 1.6
seconds if you do it one client at a time, and 0.2 seconds if you do it
properly.

The blocking design does it one client at a time, because `time.sleep`, or a
synchronous `model(x)`, does not yield. `asyncio.gather` looks concurrent and
is not: the first coroutine to touch the model owns the process until it is
finished.

The damage lands on **time to first token**, which is what a user watching a
cursor actually experiences.

In [ ]:
plt.figure(figsize=(7.5,4.4))
for name, (wall, ttft, done) in results.items():
  plt.plot(np.arange(1, len(ttft)+1)/len(ttft)*100, ttft, 'o-', label=name)
plt.yscale('log')
plt.xlabel('Percentile of clients'); plt.ylabel('Time to first token (s)')
plt.title('The same work, three ways of arranging it')
plt.legend(); plt.grid(alpha=.3); plt.show()

b = results['blocking'][1]; e = results['engine'][1]
print(f'blocking p99 TTFT is {b[-1]/e[-1]:.0f}x the engine loop')

### The threaded version looks fine here, and will not be

A thread pool keeps the event loop responsive, and in this simulation it is
as fast as the engine loop, because `time.sleep` releases the interpreter
lock and eight threads really do sleep at once.

A GPU is not like that. There is one device. Eight threads each running a
forward pass do **not** run at once: they serialise on the GPU, and worse,
each one reads the entire model from HBM to produce its single token.

That is Part 1's arithmetic again. Threads fix responsiveness. Only batching
fixes throughput, and batching needs the thing an engine loop has and a
thread pool does not: **one place where all the in-flight requests are
visible at the same time**.

So the shape is forced:

- one loop calls `step()` forever, over whatever is running right now
- requests arrive from HTTP and are put into a queue
- tokens leave through per-request queues
- nothing blocks in either direction

# What happens when a client hangs up

A user closes the tab. The request is still running, still holding KV blocks,
still costing a slot.

This is not a rare case: it is what a browser does on every navigation, and
it is the difference between a server that survives a day of real traffic and
one that fills up.

In [ ]:
async def with_cancellation(n=6, tokens=30, quit_after=5):
  qs = {i: asyncio.Queue() for i in range(n)}
  live = set(range(n))
  freed = []

  async def engine():
    for k in range(tokens):
      await asyncio.sleep(STEP_S)
      for i in list(live):
        qs[i].put_nowait(k)

  async def client(i):
    try:
      for k in range(tokens):
        await qs[i].get()
        if i % 2 == 0 and k == quit_after:
          raise asyncio.CancelledError            # the tab closes
    except asyncio.CancelledError:
      live.discard(i)                             # FREE THE BLOCKS
      freed.append((i, k))
      raise

  tasks = [asyncio.create_task(client(i)) for i in range(n)]
  await asyncio.gather(engine(), *tasks, return_exceptions=True)
  return freed, live

freed, live = await with_cancellation()
print(f'clients that hung up: {[i for i,_ in freed]}')
print(f'still running:        {sorted(live)}')
print(f'\nslots reclaimed at step {freed[0][1]} instead of step 29')

The `finally` that removes the request from the running set is the whole
feature. Without it the sequence keeps generating into a queue nobody reads,
until it hits its length limit, holding blocks the entire time.

Stage 15 has a check for exactly this: disconnect a client and assert the KV
blocks come back.

    ./vc guide 15